In [ ]:
from pathlib import Path as _ProjectPath
import sys as _project_sys

_project_root = next(
    (
        p
        for p in (_ProjectPath.cwd(), *_ProjectPath.cwd().parents)
        if (p / "project_config.json").is_file()
    ),
    None,
)
if _project_root is None:
    raise FileNotFoundError(
        "Open VS Code in the organized project root before running this notebook."
    )
if str(_project_root / "tools") not in _project_sys.path:
    _project_sys.path.insert(0, str(_project_root / "tools"))
from project_paths import ProjectPaths as _ProjectPaths

_paths = _ProjectPaths(_project_root, 'rmhd')

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
import shap
import lime
from lime.lime_text import LimeTextExplainer


sns.set(style="whitegrid", palette="muted", font_scale=1.2)
plt.rcParams["figure.figsize"] = (10, 5)


In [ ]:
data_path = _paths.rmhd_dir


df = pd.read_csv(
    _paths.input_path(os.path.join(data_path, "adhd_2019_features_tfidf_256.csv"))
)


print(df.head())
print(df.info())


print(df.isnull().sum())


In [ ]:
df = df.dropna(subset=['post', 'subreddit'])
df['post'] = df['post'].astype(str).str.strip()

print("After cleaning:", df.shape)
print(df[['subreddit', 'post']].head())


In [ ]:
import pandas as pd
import glob
import os


path = _paths.rmhd_dir


adhd_files = glob.glob(os.path.join(path, "adhd*.csv"))
print("ADHD files found:", adhd_files)


df_list = []
for file in adhd_files:
    try:
        temp = pd.read_csv(_paths.input_path(file))
        temp["source_file"] = os.path.basename(file)
        df_list.append(temp)
    except Exception as e:
        print(f"⚠️ Error reading {file}: {e}")

if df_list:
    adhd_df = pd.concat(df_list, ignore_index=True)
    print("Final ADHD dataframe shape:", adhd_df.shape)
    print("Columns:", adhd_df.columns.tolist()[:15], "...")
else:
    print("⚠️ No ADHD CSVs were loaded")


In [ ]:
import os
import glob
import pandas as pd


data_path = _paths.rmhd_dir


all_files = glob.glob(os.path.join(data_path, "*.csv"))
print("Total CSV files found:", len(all_files))

df_list = []

for file in all_files:
    try:

        disorder = os.path.basename(file).split("_")[0].lower()

        temp = pd.read_csv(_paths.input_path(file))
        temp["source_file"] = os.path.basename(file)
        temp["disorder"] = disorder
        df_list.append(temp)

    except Exception as e:
        print(f"⚠️ Error reading {file}: {e}")


if df_list:
    full_df = pd.concat(df_list, ignore_index=True)
    print("Final merged dataframe shape:", full_df.shape)
    print("Disorders present:", full_df["disorder"].unique())
else:
    print("⚠️ No CSVs merged. Check file path and naming pattern.")


In [ ]:
relevant_disorders = [
    "adhd",
    "autism",
    "addiction",
    "alcoholism",
    "anxiety",
    "healthanxiety",
    "socialanxiety",
    "bipolarreddit",
    "bpd",
    "depression",
    "ptsd",
    "schizophrenia",
    "suicidewatch",
    "lonely",
    "mentalhealth",
]

filtered_df = full_df[full_df["disorder"].isin(relevant_disorders)].reset_index(
    drop=True
)

print("Filtered dataset shape:", filtered_df.shape)
print("Disorders kept:", filtered_df["disorder"].unique())


In [ ]:
import pandas as pd


df = pd.read_csv(_paths.input_path("merged_dataset.csv"))


relevant_disorders = [
    "addiction",
    "adhd",
    "alcoholism",
    "anxiety",
    "autism",
    "bipolarreddit",
    "bpd",
    "depression",
    "healthanxiety",
    "lonely",
    "mentalhealth",
    "ptsd",
    "schizophrenia",
    "socialanxiety",
    "suicidewatch",
]


df_filtered = df[df["label"].isin(relevant_disorders)].copy()


df_filtered.reset_index(drop=True, inplace=True)


df_filtered.to_csv(_paths.output_path("filtered_dataset.csv"), index=False)

print("✅ Filtering complete. New shape:", df_filtered.shape)
print("✅ Unique labels:", df_filtered['label'].unique())


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


df = pd.read_csv(_paths.input_path("filtered_dataset.csv"))


print("Dataset Shape:", df.shape)
print("\nColumn Names:", df.columns.tolist())
print("\nMissing Values:\n", df.isnull().sum().sort_values(ascending=False).head(10))


print("\nDisorder Categories:", df['label'].unique())


plt.figure(figsize=(12, 6))
sns.countplot(
    data=df, x="label", order=df["label"].value_counts().index, palette="viridis"
)
plt.xticks(rotation=45, ha="right")
plt.title("Class Distribution of Disorders")
plt.show()


df['post_length'] = df['post'].astype(str).apply(len)

plt.figure(figsize=(10, 5))
sns.histplot(df['post_length'], bins=50, kde=True, color="teal")
plt.title("Distribution of Post Lengths (characters)")
plt.xlabel("Post Length")
plt.ylabel("Frequency")
plt.show()


if 'sent_compound' in df.columns:
    plt.figure(figsize=(10, 5))
    sns.histplot(df['sent_compound'], bins=50, kde=True, color="orange")
    plt.title("Distribution of Sentiment Compound Scores")
    plt.xlabel("Sentiment Score (-1 = Negative, +1 = Positive)")
    plt.ylabel("Frequency")
    plt.show()

    plt.figure(figsize=(12, 6))
    sns.boxplot(data=df, x='label', y='sent_compound', palette="coolwarm")
    plt.xticks(rotation=45)
    plt.title("Sentiment Distribution by Disorder Label")
    plt.xlabel("Disorder (Label)")
    plt.ylabel("Sentiment Compound Score")
    plt.show()


readability_cols = [
    'automated_readability_index',
    'coleman_liau_index',
    'flesch_kincaid_grade_level',
    'flesch_reading_ease',
    'gunning_fog_index',
    'smog_index',
]

available_readability = [col for col in readability_cols if col in df.columns]

if available_readability:
    plt.figure(figsize=(12, 6))
    df[available_readability].mean().sort_values().plot(kind="bar", color="slateblue")
    plt.title("Average Readability Scores Across Posts")
    plt.ylabel("Mean Score")
    plt.show()


num_cols = df.select_dtypes(include=['float64', 'int64']).columns
corr = df[num_cols].corr()

plt.figure(figsize=(12, 8))
sns.heatmap(corr, cmap="coolwarm", center=0)
plt.title("Correlation Heatmap of Numerical Features")
plt.show()


df['word_count'] = df['post'].astype(str).apply(lambda x: len(x.split()))

plt.figure(figsize=(12, 6))
sns.boxplot(data=df, x='label', y='word_count', palette="Set2")
plt.xticks(rotation=45)
plt.title("Word Count Distribution by Disorder Label")
plt.xlabel("Disorder (Label)")
plt.ylabel("Word Count")
plt.show()


In [ ]:
import pandas as pd


df = pd.read_csv(_paths.input_path("filtered_dataset.csv"))
print("Dataset shape:", df.shape)
print("Class distribution:\n", df['label'].value_counts())


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df['post'], df['label'], test_size=0.2, random_state=42, stratify=df['label']
)
print("Train size:", X_train.shape[0], "| Test size:", X_test.shape[0])


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=5000, stop_words='english')
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)
print("TF-IDF vectorization done. Feature shape:", X_train_vec.shape)


In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc = le.transform(y_test)

print("Label mapping:", dict(zip(le.classes_, le.transform(le.classes_))))


In [ ]:
from sklearn.metrics import classification_report, accuracy_score
import time


def train_and_evaluate(model, name):
    start = time.time()
    model.fit(X_train_vec, y_train_enc)
    y_pred = model.predict(X_test_vec)
    end = time.time()
    print(f"\n{name} Report (Time: {end-start:.2f}s):")
    print(classification_report(y_test_enc, y_pred, target_names=le.classes_))
    print(f"{name} Accuracy: {accuracy_score(y_test_enc, y_pred):.4f}")
    return end - start


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

log_reg = LogisticRegression(max_iter=1000, n_jobs=-1)


start = time.time()
log_reg.fit(X_train_vec, y_train_enc)
y_pred = log_reg.predict(X_test_vec)
end = time.time()

print(f"\nLogistic Regression Report (Time: {end-start:.2f}s):")
print(classification_report(y_test_enc, y_pred, target_names=le.classes_))
print(f"Accuracy: {accuracy_score(y_test_enc, y_pred):.4f}")


cm = confusion_matrix(y_test_enc, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=le.classes_,
    yticklabels=le.classes_,
)
plt.title("Logistic Regression Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()


In [ ]:
from sklearn.svm import LinearSVC

svm = LinearSVC(max_iter=5000)

start = time.time()
svm.fit(X_train_vec, y_train_enc)
y_pred = svm.predict(X_test_vec)
end = time.time()

print(f"\nLinear SVM Report (Time: {end-start:.2f}s):")
print(classification_report(y_test_enc, y_pred, target_names=le.classes_))
print(f"Accuracy: {accuracy_score(y_test_enc, y_pred):.4f}")

cm = confusion_matrix(y_test_enc, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Greens',
    xticklabels=le.classes_,
    yticklabels=le.classes_,
)
plt.title("Linear SVM Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()


In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=42)

start = time.time()
rf.fit(X_train_vec, y_train_enc)
y_pred = rf.predict(X_test_vec)
end = time.time()

print(f"\nRandom Forest Report (Time: {end-start:.2f}s):")
print(classification_report(y_test_enc, y_pred, target_names=le.classes_))
print(f"Accuracy: {accuracy_score(y_test_enc, y_pred):.4f}")

cm = confusion_matrix(y_test_enc, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Oranges',
    xticklabels=le.classes_,
    yticklabels=le.classes_,
)
plt.title("Random Forest Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()


In [ ]:
import time
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix


rf = RandomForestClassifier(
    n_estimators=500,
    max_depth=70,
    min_samples_leaf=2,
    max_features="sqrt",
    n_jobs=-1,
    random_state=42,
)


start = time.time()
rf.fit(X_train_vec, y_train_enc)
y_pred = rf.predict(X_test_vec)
end = time.time()


print(f"\nRandom Forest Report (Time: {end-start:.2f}s):")
print(classification_report(y_test_enc, y_pred, target_names=le.classes_))
print(f"Accuracy: {accuracy_score(y_test_enc, y_pred):.4f}")


cm = confusion_matrix(y_test_enc, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Oranges',
    xticklabels=le.classes_,
    yticklabels=le.classes_,
)
plt.title("Random Forest Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()


importances = rf.feature_importances_
indices = importances.argsort()[-15:][::-1]

plt.figure(figsize=(10, 6))
plt.barh(
    [vectorizer.get_feature_names_out()[i] for i in indices],
    importances[indices],
    color='skyblue',
)
plt.xlabel("Importance")
plt.title("Top 15 Features (Random Forest)")
plt.gca().invert_yaxis()
plt.show()


In [ ]:
import time
import joblib
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix


configs = [(600, 80), (300, 40), (400, 60)]

results = []

for n_estimators, max_depth in configs:
    print(
        f"\n=== Training RF with n_estimators={n_estimators}, max_depth={max_depth} ==="
    )

    rf = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_leaf=2,
        max_features="sqrt",
        n_jobs=-1,
        random_state=42,
    )

    start = time.time()
    rf.fit(X_train_vec, y_train_enc)
    y_pred = rf.predict(X_test_vec)
    end = time.time()

    acc = accuracy_score(y_test_enc, y_pred)
    report = classification_report(
        y_test_enc, y_pred, target_names=le.classes_, output_dict=True
    )
    precision = report["macro avg"]["precision"]
    recall = report["macro avg"]["recall"]

    model_path = _paths.output_path(f'rf_{n_estimators}_{max_depth}.joblib')
    joblib.dump(rf, _paths.output_path(model_path))

    import os

    size_mb = os.path.getsize(model_path) / (1024 * 1024)

    results.append(
        {
            "n_estimators": n_estimators,
            "max_depth": max_depth,
            "accuracy": acc,
            "precision": precision,
            "recall": recall,
            "time_sec": end - start,
            "model_size_MB": size_mb,
        }
    )

    cm = confusion_matrix(y_test_enc, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(
        cm,
        annot=False,
        fmt='d',
        cmap='Oranges',
        xticklabels=le.classes_,
        yticklabels=le.classes_,
    )
    plt.title(f"RF Confusion Matrix (n_estimators={n_estimators}, depth={max_depth})")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.show()


df_results = pd.DataFrame(results)
print("\n=== Benchmark Results ===")
print(df_results)


plt.figure(figsize=(8, 5))
plt.scatter(df_results["model_size_MB"], df_results["accuracy"], s=100, c="blue")
for i, row in df_results.iterrows():
    plt.text(
        row["model_size_MB"] + 10,
        row["accuracy"],
        f"{row['n_estimators']}/{row['max_depth']}",
        fontsize=9,
    )
plt.xlabel("Model Size (MB)")
plt.ylabel("Accuracy")
plt.title("Random Forest Accuracy vs Model Size")
plt.show()


In [ ]:
import joblib


joblib.dump(rf, _paths.output_path("random_forest_model.pkl"))
joblib.dump(svm, _paths.output_path("linear_svm_model.pkl"))
joblib.dump(log_reg, _paths.output_path("logistic_regression_model.pkl"))
joblib.dump(vectorizer, _paths.output_path("tfidf_vectorizer.pkl"))
joblib.dump(le, _paths.output_path("label_encoder.pkl"))

print("All models and encoders have been saved successfully!")


In [ ]:
import pandas as pd
import numpy as np
import joblib
import shap
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer


print("Loading models and vectorizer...")

rf = joblib.load(_paths.input_path("random_forest_model.pkl"))
svm = joblib.load(_paths.input_path("linear_svm_model.pkl"))
log_reg = joblib.load(_paths.input_path("logistic_regression_model.pkl"))
vectorizer = joblib.load(_paths.input_path("tfidf_vectorizer.pkl"))

print("Models and vectorizer loaded successfully!")


print("Loading dataset...")

df = pd.read_csv(_paths.input_path("filtered_dataset.csv"))

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import shap


class_mapping = {
    0: "addiction",
    1: "adhd",
    2: "alcoholism",
    3: "anxiety",
    4: "autism",
    5: "bipolarreddit",
    6: "bpd",
    7: "depression",
    8: "healthanxiety",
    9: "lonely",
    10: "mentalhealth",
    11: "ptsd",
    12: "schizophrenia",
    13: "socialanxiety",
    14: "suicidewatch",
}


def explain_prediction(input_text):
    X_input = vectorizer.transform([input_text])
    feature_names = vectorizer.get_feature_names_out()

    prob_rf = rf.predict_proba(X_input)[0]
    prob_log = log_reg.predict_proba(X_input)[0]
    prob_svm = np.zeros_like(prob_rf)
    try:
        prob_svm = svm.predict_proba(X_input)[0]
    except:
        prob_svm = np.zeros_like(prob_rf)

    max_rf, max_log, max_svm = np.max(prob_rf), np.max(prob_log), np.max(prob_svm)
    best_model_name, best_model, best_probs = "Random Forest", rf, prob_rf
    if max_log > max_rf and max_log >= max_svm:
        best_model_name, best_model, best_probs = (
            "Logistic Regression",
            log_reg,
            prob_log,
        )
    elif max_svm > max_rf and max_svm >= max_log:
        best_model_name, best_model, best_probs = "Linear SVM", svm, prob_svm

    pred_idx = np.argmax(best_probs)
    pred_class = best_model.classes_[pred_idx]
    pred_name = class_mapping[pred_class]
    pred_score = best_probs[pred_idx] * 100

    named_probs = {
        class_mapping[k]: float(v) for k, v in zip(best_model.classes_, best_probs)
    }
    named_probs_sorted = dict(
        sorted(named_probs.items(), key=lambda x: x[1], reverse=True)
    )

    print(f"Best Model Used: {best_model_name}")
    print(f"Predicted Condition: {pred_name} ({pred_score:.2f}%)")
    print("Prediction Probabilities (top 5):")
    for cls, prob in list(named_probs_sorted.items())[:5]:
        marker = "<-- Main Prediction" if cls == pred_name else ""
        print(f"  {cls}: {prob:.3f} {marker}")

    top_n = 10
    nonzero_idx = X_input.nonzero()[1]
    if len(nonzero_idx) == 0:
        print("No meaningful features found for explanation.")
        return

    if best_model_name == "Random Forest":

        importances = best_model.feature_importances_

        local_importances = importances[nonzero_idx]

        n_top = min(top_n, len(nonzero_idx))
        top_idx = np.argsort(local_importances)[-n_top:]

        top_features = feature_names[nonzero_idx][top_idx]
        top_values = local_importances[top_idx]

        plt.figure(figsize=(10, 6))
        plt.barh(top_features[::-1], top_values[::-1], color="skyblue")
        plt.xlabel("Feature Importance (local input)")
        plt.title(
            f"Top Features contributing to '{pred_name}' (Random Forest, Feature Importance)"
        )
        plt.show()

        global_importances = best_model.feature_importances_
        n_top_heat = 10
        top_features_idx = np.argsort(global_importances)[-n_top_heat:]
        top_features_global = feature_names[top_features_idx]
        top_values_global = global_importances[top_features_idx]

        plt.figure(figsize=(10, 6))
        plt.imshow(
            np.tile(top_values_global, (len(best_model.classes_), 1)),
            cmap="coolwarm",
            aspect="auto",
        )
        plt.xticks(range(n_top_heat), top_features_global, rotation=45, ha="right")
        plt.yticks(
            range(len(best_model.classes_)),
            [class_mapping[c] for c in best_model.classes_],
        )
        plt.colorbar(label="Feature Importance")
        plt.title("Global Feature Importances Across All Classes (Random Forest)")
        plt.show()

    else:
        coefs = best_model.coef_
        shap_vals = coefs[pred_idx, nonzero_idx]
        n_top = min(top_n, len(nonzero_idx))
        top_idx = np.argsort(np.abs(shap_vals))[-n_top:]
        top_features = feature_names[nonzero_idx][top_idx]
        top_values = shap_vals[top_idx]

        plt.figure(figsize=(10, 6))
        colors = ['green' if v > 0 else 'red' for v in top_values]
        plt.barh(top_features[::-1], top_values[::-1], color=colors)
        plt.xlabel("Feature Contribution")
        plt.title(f"Top Features contributing to '{pred_name}' ({best_model_name})")
        plt.show()

        all_coefs = coefs[:, nonzero_idx]
        n_top_heat = min(top_n, all_coefs.shape[1])
        top_features_idx = np.argsort(np.abs(all_coefs).mean(axis=0))[-n_top_heat:]
        plt.figure(figsize=(10, 6))
        plt.imshow(all_coefs[:, top_features_idx], cmap="coolwarm", aspect="auto")
        plt.xticks(
            range(n_top_heat),
            feature_names[nonzero_idx][top_features_idx],
            rotation=45,
            ha="right",
        )
        plt.yticks(
            range(len(best_model.classes_)),
            [class_mapping[c] for c in best_model.classes_],
        )
        plt.colorbar(label="Feature Contribution")
        plt.title(f"Feature Contributions Across All Classes ({best_model_name})")
        plt.show()

    summary_text = f"\nBased on the input text, the model predicts '{pred_name}' with {pred_score:.2f}% confidence.\nTop contributing words/features: {', '.join(top_features[::-1])}"
    print(summary_text)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from lime.lime_text import LimeTextExplainer
from IPython.display import display, HTML


models = {"Logistic Regression": log_reg, "Linear SVM": svm, "Random Forest": rf}

class_names_all = [class_mapping[i] for i in sorted(class_mapping.keys())]


def explain_prediction(input_text, top_n=12, show_plots=True):
    """
    Single-call explain function:
     - auto-selects best model by max probability,
     - for LR / Linear SVM: exact linear contributions (coef * tfidf) + LIME,
     - for Random Forest: local/global feature_importances + LIME (no SHAP).
    Requires: vectorizer, log_reg, svm, rf, class_mapping in globals.
    """

    X_input = vectorizer.transform([input_text])
    feature_names = vectorizer.get_feature_names_out()

    best_model_name = None
    best_model = None
    best_probs = None
    best_score = -1.0
    for name, m in models.items():
        try:
            probs = m.predict_proba(X_input)[0]
        except Exception:
            probs = np.zeros(len(class_mapping))
        score = float(np.max(probs))
        if score > best_score:
            best_score = score
            best_model_name = name
            best_model = m
            best_probs = probs

    if best_model is None:
        print("No model available to predict.")
        return

    model_classes = list(best_model.classes_.astype(int))
    model_class_names = [class_mapping[int(c)] for c in model_classes]

    pred_idx_in_model = int(np.argmax(best_probs))
    pred_global_idx = int(best_model.classes_[pred_idx_in_model])
    pred_name = class_mapping[pred_global_idx]
    pred_score = best_probs[pred_idx_in_model]

    print("=" * 90)
    print(
        "Input (trim):",
        (input_text[:260] + "...") if len(input_text) > 260 else input_text,
    )
    print("Best model:", best_model_name)
    print(f"Predicted: {pred_name}  (confidence = {pred_score:.4f})")
    print("\nTop probabilities (model order):")
    order = np.argsort(best_probs)[::-1]
    for i in order[:10]:
        gname = class_mapping[int(best_model.classes_[i])]
        mark = "<-- predicted" if i == pred_idx_in_model else ""
        print(f"  {gname:25s} {best_probs[i]:.4f} {mark}")

    if show_plots:
        plt.figure(figsize=(9, 6))
        sns.barplot(
            x=best_probs[order],
            y=[class_mapping[int(best_model.classes_[i])] for i in order],
            palette="viridis",
        )
        plt.xlabel("Probability")
        plt.title(f"Prediction probabilities ({best_model_name})")
        plt.show()

    def predict_proba_for_lime(texts):
        return best_model.predict_proba(vectorizer.transform(texts))

    if best_model_name == "Random Forest":
        print(
            "\n-- Random Forest explanation (no SHAP). Using feature_importances_ + LIME."
        )

        importances = best_model.feature_importances_
        nz_idx = X_input.nonzero()[1]
        if nz_idx.size:
            local_vals = importances[nz_idx]
            top_k = min(top_n, len(local_vals))
            top_order = np.argsort(local_vals)[-top_k:]
            top_feats_idx = nz_idx[top_order]
            top_feats = feature_names[top_feats_idx]
            top_vals = local_vals[top_order]
            print("\nTop local features (present in input):")
            for f, v in zip(top_feats[::-1], top_vals[::-1]):
                print(f"  {f:30s} {v:.6f}")
            if show_plots:
                plt.figure(figsize=(9, 5))
                plt.barh(top_feats[::-1], top_vals[::-1], color="skyblue")
                plt.title("Local Feature Importances (Random Forest)")
                plt.xlabel("Importance")
                plt.show()
        else:
            print("  (No non-zero features in vectorized input)")

        try:
            top_idx_global = np.argsort(importances)[-min(25, len(importances)) :]
            top_feats_global = feature_names[top_idx_global]
            top_vals_global = importances[top_idx_global]
            print("\nTop global features (Random Forest):")
            for f, v in zip(top_feats_global[::-1], top_vals_global[::-1]):
                print(f"  {f:30s} {v:.6f}")
            if show_plots:
                plt.figure(figsize=(10, 6))
                plt.barh(top_feats_global[::-1], top_vals_global[::-1], color="orange")
                plt.title("Global Feature Importances (Random Forest)")
                plt.xlabel("Importance")
                plt.show()
        except Exception as e:
            print("  Could not compute global RF importances:", e)

        try:
            lime_explainer = LimeTextExplainer(class_names=model_class_names)
            exp = lime_explainer.explain_instance(
                input_text, predict_proba_for_lime, num_features=top_n
            )
            print("\nLIME top contributions (word, weight):")
            for w, wt in exp.as_list():
                print(f"  {w:30s} {wt:+.4f}")

            try:
                display(HTML(exp.as_html()))
            except Exception:
                pass
            if show_plots:
                fig = exp.as_pyplot_figure()
                fig.suptitle("LIME contributions (Random Forest)", fontsize=14)
                plt.show()
        except Exception as e:
            print("⚠️ LIME failed for Random Forest:", e)

        print(
            "\nSummary: RF explained by feature_importances_ (global+local) and LIME."
        )
        print("=" * 90)
        return

    print(
        "\n-- Linear model branch (Logistic Regression / Linear SVM). Using exact linear contributions + LIME."
    )

    try:
        x_dense = X_input.toarray()[0]
        coefs = best_model.coef_
        coef_row = coefs[pred_idx_in_model]
        contrib_vector = coef_row * x_dense
        nz = np.where(x_dense != 0)[0]
        if nz.size == 0:
            print("  No non-zero tf-idf features for this input (nothing to explain).")
        else:
            local_pairs = [(feature_names[i], float(contrib_vector[i])) for i in nz]
            local_pairs_sorted = sorted(
                local_pairs, key=lambda x: abs(x[1]), reverse=True
            )[:top_n]
            print(
                "\nTop local contributions (feature: contribution) — exact for linear model:"
            )
            for f, v in local_pairs_sorted:
                sign = "+" if v > 0 else "-"
                print(f"  {f:30s} {sign}{abs(v):.6f}")
            if show_plots:
                feats, vals = zip(*local_pairs_sorted)
                plt.figure(figsize=(9, 6))
                plt.barh(
                    feats[::-1],
                    vals[::-1],
                    color=["green" if v > 0 else "red" for v in vals[::-1]],
                )
                plt.title("Top local linear contributions (coef * tfidf)")
                plt.xlabel("Contribution (coef * tfidf)")
                plt.show()
    except Exception as e:
        print("  Failed to compute linear contributions:", e)

    try:
        lime_explainer = LimeTextExplainer(class_names=model_class_names)
        exp = lime_explainer.explain_instance(
            input_text, predict_proba_for_lime, num_features=top_n
        )
        print("\nLIME top contributions (word, weight):")
        for w, wt in exp.as_list():
            print(f"  {w:30s} {wt:+.4f}")
        try:
            display(HTML(exp.as_html()))
        except Exception:
            pass
        if show_plots:
            fig = exp.as_pyplot_figure()
            fig.suptitle("LIME contributions (linear model)", fontsize=14)
            plt.show()
    except Exception as e:
        print("⚠️ LIME failed for linear model:", e)

    try:
        print("\nGlobal top features (model-level):")
        if hasattr(best_model, "coef_"):
            importances = np.mean(np.abs(best_model.coef_), axis=0)
            top_idx = np.argsort(importances)[-min(25, len(importances)) :]
            for f, v in zip(feature_names[top_idx[::-1]], importances[top_idx[::-1]]):
                print(f"  {f:30s} {v:.6f}")
            if show_plots:
                feats = feature_names[top_idx]
                vals = importances[top_idx]
                plt.figure(figsize=(10, 6))
                plt.barh(feats[::-1], vals[::-1], color="orange")
                plt.title("Global top features (linear model)")
                plt.xlabel("avg |coef|")
                plt.show()
    except Exception as e:
        print("⚠️ global feature extraction failed:", e)

    try:
        top_words = [w for w, _ in (exp.as_list()[:5] if 'exp' in locals() else [])]
        if top_words:
            print(
                "\nSummary: model decision was driven by words (LIME top):",
                ", ".join(top_words),
            )
        else:
            print(
                "\nSummary: local top features printed above (coef * tfidf) indicate which words/features drove the prediction."
            )
    except Exception:
        pass

    print("=" * 90)


In [ ]:
text_input = """Lately I can’t sleep, my thoughts keep racing, and even small noises make me panic. Sometimes I feel okay, but then suddenly my mood drops for no reason."""
explain_prediction(text_input)


In [ ]:
text_input = """
Over the past few months, I have been feeling an increasing sense of anxiety and unease. 
Simple daily activities, such as getting out of bed, responding to messages, or preparing meals, 
have become overwhelming tasks. I struggle to concentrate at work, and I often feel mentally exhausted 
by midday. Sleep has become irregular—I either lie awake worrying for hours or wake up multiple times during the night. 
I notice that I avoid social gatherings and have started withdrawing from friends and family. 
Hobbies and activities I used to enjoy no longer bring me satisfaction. 
Physical symptoms, such as a racing heartbeat, tightness in my chest, and tension headaches, 
often accompany my anxious thoughts. I frequently ruminate over past events and worry excessively about the future, 
feeling hopeless and unsure of how to manage even basic responsibilities. 
These feelings leave me drained, frustrated, and increasingly isolated, creating a cycle that I find difficult to break.
"""
explain_prediction(text_input)


In [ ]:
text_input = """
I often find myself unable to focus on a single task for long periods. 
Even when something is important, I get distracted easily by noises, thoughts, or messages on my phone. 
I start projects but rarely finish them, and I forget deadlines or appointments. 
I feel restless and fidgety, tapping my hands or moving around when I should be sitting still. 
I impulsively say or do things without thinking, and it often gets me into trouble at work or school. 
Even when I try to organize myself, my thoughts jump quickly from one idea to another.
"""
explain_prediction(text_input)


In [ ]:
text_input = """
Some days I feel unstoppable, energized, and overly confident, taking impulsive risks. 
Other days, I’m lethargic, hopeless, and unable to complete basic tasks. 
My mood swings unpredictably, leaving me exhausted and confused.
"""
explain_prediction(text_input)


In [ ]:
text_input = """

"""
explain_prediction(text_input)


In [ ]:
import zipfile

zip_path = _paths.input_path('mental-health-dataset.zip')
extract_path = _paths.output_subdir('mental_health_data')

with zipfile.ZipFile(_paths.input_path(zip_path), 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Dataset extracted successfully!")


In [ ]:
import pandas as pd
import numpy as np
import re
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter


In [ ]:
dataset_path = _paths.input_path('mental_health_data/mental_disorders_reddit.csv')
df = pd.read_csv(_paths.input_path(dataset_path))

print("Dataset loaded successfully")
print("Shape of dataset:", df.shape)
print("Columns:", df.columns.tolist())


In [ ]:
def clean_text(text):
    if pd.isnull(text):
        return ""
    text = text.lower()
    text = re.sub(r"http\S+|www.\S+", "", text)
    text = re.sub(r"[^a-z\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


df["text"] = (df["title"].fillna("") + " " + df["selftext"].fillna("")).apply(
    clean_text
)

print("🔹 New text column sample:\n", df["text"].head())


In [ ]:
vectorizer = joblib.load(_paths.input_path("tfidf_vectorizer.pkl"))
log_reg = joblib.load(_paths.input_path("logistic_regression_model.pkl"))
svm = joblib.load(_paths.input_path("linear_svm_model.pkl"))
rf = joblib.load(_paths.input_path("random_forest_model.pkl"))
label_encoder = joblib.load(_paths.input_path("label_encoder.pkl"))

print("✅ Vectorizer & Models loaded successfully")
print("Classes:", label_encoder.classes_)


In [ ]:
X = vectorizer.transform(df["text"])

print("✅ Vectorization complete")
print("Shape of vectorized data:", X.shape)


In [ ]:
print("🔄 Running predictions... (this may take time)")

pred_log = log_reg.predict(X)
pred_svm = svm.predict(X)
pred_rf = rf.predict(X)


pred_log_labels = label_encoder.inverse_transform(pred_log)
pred_svm_labels = label_encoder.inverse_transform(pred_svm)
pred_rf_labels = label_encoder.inverse_transform(pred_rf)

print("✅ Predictions complete")


In [ ]:
from collections import Counter
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np


def trend_analysis(df, model, pred_labels, model_name):
    counts = Counter(pred_labels)
    total = sum(counts.values())
    percentages = {d: (n / total) * 100 for d, n in counts.items()}

    plt.figure(figsize=(8, 8))
    wedges, _, _ = plt.pie(
        counts.values(),
        autopct='%1.1f%%',
        startangle=140,
        colors=sns.color_palette("tab20"),
    )

    plt.title(f"Overall Disease Trend - {model_name}", fontsize=14)
    plt.legend(
        wedges,
        [f"{d} ({n}, {percentages[d]:.1f}%)" for d, n in counts.items()],
        title="Disorders",
        loc="center left",
        bbox_to_anchor=(1, 0, 0.5, 1),
        fontsize=10,
    )
    plt.show()

    plt.figure(figsize=(12, 6))
    sns.barplot(x=list(counts.keys()), y=list(counts.values()), palette="tab20")
    plt.xticks(rotation=45, ha="right")
    plt.ylabel("Number of Cases")
    plt.title(f"Cases per Disease - {model_name}")
    plt.show()

    feature_names = vectorizer.get_feature_names_out()
    top_n = 15

    if hasattr(model, "coef_"):
        global_importances = np.mean(np.abs(model.coef_), axis=0)
    elif hasattr(model, "feature_importances_"):
        global_importances = model.feature_importances_
    else:
        print("⚠️ No feature importances available for this model.")
        return

    top_idx = np.argsort(global_importances)[-top_n:]
    top_features = feature_names[top_idx]
    top_values = global_importances[top_idx]

    plt.figure(figsize=(10, 6))
    plt.barh(top_features, top_values, color="skyblue")
    plt.xlabel("Importance / Contribution")
    plt.title(f"Top {top_n} Features Driving Predictions - {model_name} (Global)")
    plt.show()

    print("\n🔍 Global Explanation Summary:")
    for feat, val in zip(top_features[::-1], top_values[::-1]):
        print(f" - {feat}: {val:.4f}")

    print(f"\n📑 Top Features Per Class ({model_name}):")
    if hasattr(model, "coef_"):
        for i, cls in enumerate(model.classes_):
            class_importances = np.abs(model.coef_[i])
            top_idx_cls = np.argsort(class_importances)[-top_n:]
            top_feats_cls = feature_names[top_idx_cls]
            top_vals_cls = class_importances[top_idx_cls]

            print(f"\n🔹 {label_encoder.inverse_transform([cls])[0]}:")
            for feat, val in zip(top_feats_cls[::-1], top_vals_cls[::-1]):
                print(f"   - {feat}: {val:.4f}")

    elif hasattr(model, "feature_importances_"):
        print("Random Forest does not provide per-class feature importances directly.")
        print("Using global feature importances instead.")


In [ ]:
def trend_analysis(pred_labels, model_name):

    counts = Counter(pred_labels)
    total = sum(counts.values())
    percentages = {d: (n / total) * 100 for d, n in counts.items()}

    plt.figure(figsize=(8, 8))
    plt.pie(
        counts.values(),
        labels=[f"{d} ({p:.1f}%)" for d, p in percentages.items()],
        autopct='%1.1f%%',
        startangle=140,
        colors=sns.color_palette("tab20"),
    )
    plt.title(f"Overall Disease Trend - {model_name}")
    plt.show()

    plt.figure(figsize=(12, 6))
    sns.barplot(x=list(counts.keys()), y=list(counts.values()), palette="tab20")
    plt.xticks(rotation=45, ha="right")
    plt.ylabel("Number of Cases")
    plt.title(f"Cases per Disease - {model_name}")
    plt.show()

    print(f"\n📊 Disease Trend Analysis ({model_name}):")
    for d, n in counts.items():
        print(f" - {d}: {n} cases ({percentages[d]:.2f}%)")


trend_analysis(pred_log_labels, "Logistic Regression")
trend_analysis(pred_svm_labels, "Linear SVM")
trend_analysis(pred_rf_labels, "Random Forest")


In [ ]:
rf = RandomForestClassifier(
    n_estimators=200,
    n_jobs=-1,
    random_state=42,
    class_weight='balanced',
    max_features='sqrt',
)


In [ ]:
import joblib
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt


svm = joblib.load(_paths.input_path("linear_svm_model.pkl"))
lr = joblib.load(_paths.input_path("logistic_regression_model.pkl"))


In [ ]:
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report


df = pd.read_csv(_paths.input_path("filtered_dataset.csv"))


X_train, X_test, y_train, y_test = train_test_split(
    df["post"], df["label"], test_size=0.2, random_state=42, stratify=df["label"]
)


vectorizer = joblib.load(_paths.input_path("tfidf_vectorizer.pkl"))
le = joblib.load(_paths.input_path("label_encoder.pkl"))


X_test_vec = vectorizer.transform(X_test)
y_test_enc = le.transform(y_test)


svm = joblib.load(_paths.input_path("linear_svm_model.pkl"))
lr = joblib.load(_paths.input_path("logistic_regression_model.pkl"))
rf = joblib.load(_paths.input_path("random_forest_model.pkl"))


models = {"Linear SVM": svm, "Logistic Regression": lr, "Random Forest": rf}

for name, model in models.items():

    y_pred = model.predict(X_test_vec)

    accuracy = accuracy_score(y_test_enc, y_pred)
    cm = confusion_matrix(y_test_enc, y_pred)

    print("\n" + "=" * 60)
    print(name)
    print("=" * 60)

    print(f"Accuracy: {accuracy:.6f}")
    print(f"Accuracy: {accuracy * 100:.2f}%")

    print("\nClassification Report:")
    print(classification_report(y_test_enc, y_pred, target_names=le.classes_))

    plt.figure(figsize=(15, 13), dpi=300)

    ax = sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=le.classes_,
        yticklabels=le.classes_,
        annot_kws={"size": 10, "weight": "normal"},
        linewidths=0.3,
        linecolor="white",
        cbar_kws={"shrink": 0.9},
    )

    ax.set_title(f"{name} - Confusion Matrix", fontsize=18, fontweight="bold", pad=15)

    ax.set_xlabel("Predicted", fontsize=14, fontweight="bold", labelpad=10)

    ax.set_ylabel("Actual", fontsize=14, fontweight="bold", labelpad=10)

    ax.set_xticklabels(ax.get_xticklabels(), fontsize=11, rotation=45, ha="right")

    ax.set_yticklabels(ax.get_yticklabels(), fontsize=11, rotation=0)

    cbar = ax.collections[0].colorbar
    cbar.ax.tick_params(labelsize=10)

    plt.tight_layout()

    filename = _paths.output_path(
        name.lower().replace(' ', '_') + '_confusion_matrix.png'
    )

    plt.savefig(_paths.output_path(filename), dpi=600, bbox_inches="tight")

    plt.show()
    plt.close()
    plt.show()


In [ ]:
plt.figure(figsize=(17, 15), dpi=300)

ax = sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    square=True,
    annot_kws={"size": 14, "weight": "bold"},
    xticklabels=le.classes_,
    yticklabels=le.classes_,
    linewidths=0.4,
    linecolor="white",
    cbar_kws={"shrink": 0.88},
)


ax.set_title(f"{name} - Confusion Matrix", fontsize=22, fontweight="bold", pad=20)


ax.set_xlabel("Predicted", fontsize=17, fontweight="bold", labelpad=12)

ax.set_ylabel("Actual", fontsize=17, fontweight="bold", labelpad=12)


ax.set_yticklabels(
    ax.get_yticklabels(), fontsize=14, fontweight="bold", rotation=0, va="center"
)


ax.set_xticklabels(
    ax.get_xticklabels(), fontsize=13, fontweight="bold", rotation=45, ha="right"
)


cbar = ax.collections[0].colorbar
cbar.ax.tick_params(labelsize=11)


plt.tight_layout()


filename = _paths.output_path(name.lower().replace(' ', '_') + '_confusion_matrix.png')

plt.savefig(_paths.output_path(filename), dpi=600, bbox_inches="tight")

plt.show()
plt.close()


In [ ]:
plt.figure(figsize=(17, 15), dpi=300)

ax = sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    square=True,
    annot_kws={"size": 16, "weight": "bold"},
    xticklabels=le.classes_,
    yticklabels=le.classes_,
    linewidths=0.6,
    linecolor="black",
    cbar_kws={"shrink": 0.88},
)


ax.set_title(f"{name} - Confusion Matrix", fontsize=22, fontweight="bold", pad=20)


ax.set_xlabel("Predicted", fontsize=18, fontweight="bold", labelpad=12)

ax.set_ylabel("Actual", fontsize=18, fontweight="bold", labelpad=12)


ax.set_yticklabels(
    ax.get_yticklabels(), fontsize=15, fontweight="bold", rotation=0, va="center"
)


ax.set_xticklabels(
    ax.get_xticklabels(),
    fontsize=15,
    fontweight="bold",
    rotation=90,
    ha="center",
    va="top",
)


ax.tick_params(axis="both", which="major", length=5, width=1.2, direction="out")


cbar = ax.collections[0].colorbar
cbar.ax.tick_params(labelsize=12, width=1)


plt.tight_layout()

filename = _paths.output_path(name.lower().replace(' ', '_') + '_confusion_matrix.png')

plt.savefig(_paths.output_path(filename), dpi=600, bbox_inches="tight")

plt.show()
plt.close()


In [ ]:
y_pred = svm.predict(X_test_vec)
cm = confusion_matrix(y_test_enc, y_pred)

plt.figure(figsize=(17, 15), dpi=300)

ax = sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    square=True,
    annot_kws={"size": 16, "weight": "bold"},
    xticklabels=le.classes_,
    yticklabels=le.classes_,
    linewidths=0.6,
    linecolor="black",
    cbar_kws={"shrink": 0.88},
)

ax.set_title("Linear SVM - Confusion Matrix", fontsize=22, fontweight="bold", pad=20)

ax.set_xlabel("Predicted", fontsize=18, fontweight="bold", labelpad=12)

ax.set_ylabel("Actual", fontsize=18, fontweight="bold", labelpad=12)


ax.set_yticklabels(
    ax.get_yticklabels(), fontsize=15, fontweight="bold", rotation=0, va="center"
)


ax.set_xticklabels(
    ax.get_xticklabels(),
    fontsize=15,
    fontweight="bold",
    rotation=90,
    ha="center",
    va="top",
)


ax.tick_params(axis="both", which="major", length=5, width=1.2, direction="out")


cbar = ax.collections[0].colorbar
cbar.ax.tick_params(labelsize=12, width=1)

plt.tight_layout()

plt.savefig(
    _paths.output_path("linear_svm_confusion_matrix.png"), dpi=600, bbox_inches="tight"
)

plt.show()
plt.close()


In [ ]:
y_pred = svm.predict(X_test_vec)
cm = confusion_matrix(y_test_enc, y_pred)

plt.figure(figsize=(17, 15), dpi=300)

ax = sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    square=True,
    annot_kws={"size": 16, "weight": "bold"},
    xticklabels=le.classes_,
    yticklabels=le.classes_,
    linewidths=0,
    cbar_kws={"shrink": 0.88},
)


ax.set_title("Linear SVM - Confusion Matrix", fontsize=22, fontweight="bold", pad=20)


ax.set_xlabel("Predicted", fontsize=18, fontweight="bold", labelpad=12)

ax.set_ylabel("Actual", fontsize=18, fontweight="bold", labelpad=12)


ax.set_yticklabels(
    ax.get_yticklabels(), fontsize=16, fontweight="normal", rotation=0, va="center"
)


ax.set_xticklabels(
    ax.get_xticklabels(),
    fontsize=16,
    fontweight="normal",
    rotation=90,
    ha="center",
    va="top",
)


ax.tick_params(axis="both", which="major", length=5, width=1.2, direction="out")


cbar = ax.collections[0].colorbar
cbar.ax.tick_params(labelsize=12, width=1)

plt.tight_layout()

plt.savefig(
    _paths.output_path("linear_svm_confusion_matrix.png"), dpi=600, bbox_inches="tight"
)

plt.show()
plt.close()
